# Eight repeated operations (self and fixed loops) vs Texture Mixer (`tab:app-repeat`)

Produces `degradation_5_9/results/*`; the autoencoder-only row is printed only, everything else is traceable to that file.

**Needs:** DTD, the CNT checkpoint, the Texture Mixer checkpoint.

**Maintained runnable path:** none. `score_only.py` does NOT currently check `tab:app-repeat`; run this notebook as-is to reproduce or inspect it.

In [ ]:
# ============================================================
# PHASE 1, CELL 1 — TensorFlow only. Loads Texture Mixer.
# ============================================================
import os, sys, types, importlib, importlib.util, importlib.machinery, glob, random, time
import numpy as np, pandas as pd
from PIL import Image, ImageFile
import matplotlib.pyplot as plt
ImageFile.LOAD_TRUNCATED_IMAGES = True
from google.colab import drive
drive.mount('/content/drive')

BASE = os.environ.get('DATA_ROOT', '/content/drive/MyDrive/DATA_ROOT')  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
TM_STORE = f'{BASE}/texturemixer'
DEG = f'{BASE}/degradation_5_9'; os.makedirs(DEG, exist_ok=True)

os.chdir('/content')
if not os.path.isdir('/content/TextureMixer'):
    !git clone -q https://github.com/ningyu1991/TextureMixer.git
if not os.path.isdir('/content/TextureMixer/tensorflow_vgg'):
    !git clone -q https://github.com/machrisaa/tensorflow-vgg.git /content/TextureMixer/tensorflow_vgg
os.chdir('/content/TextureMixer'); sys.path.insert(0,'/content/TextureMixer')

if 'imp' not in sys.modules:
    _imp = types.ModuleType('imp')
    def _ls(n,p,f=None):
        s=importlib.util.spec_from_file_location(n,p); m=importlib.util.module_from_spec(s)
        sys.modules[n]=m; s.loader.exec_module(m); return m
    def _fm(n,p=None):
        s=importlib.machinery.PathFinder().find_spec(n,p)
        if s is None: raise ImportError(n)
        return (None,s.origin,('.py','r',1))
    _imp.load_source=_ls; _imp.find_module=_fm; _imp.new_module=lambda n: types.ModuleType(n)
    _imp.reload=importlib.reload; _imp.acquire_lock=lambda:None
    _imp.release_lock=lambda:None; _imp.PY_SOURCE=1
    sys.modules['imp']=_imp

import tensorflow as _tf2
print('TF', _tf2.__version__, '| GPU:', _tf2.config.list_physical_devices('GPU'))
tf1=_tf2.compat.v1; tf1.disable_eager_execution(); tf1.disable_v2_behavior()
tf1.logging.set_verbosity(tf1.logging.ERROR)
if not hasattr(tf1,'Dimension'): tf1.Dimension=getattr(_tf2,'Dimension',int)
!pip install -q tf_slim
import tf_slim as _slim
_c=types.ModuleType('tensorflow.contrib'); _c.slim=_slim; _c.framework=_slim; _c.layers=_slim
sys.modules['tensorflow.contrib']=_c
for n in ['tensorflow.contrib.slim','tensorflow.contrib.framework','tensorflow.contrib.layers']:
    sys.modules[n]=_slim
tf1.contrib=_c; sys.modules['tensorflow']=tf1

_p='/content/TextureMixer/tfutil.py'; _s=open(_p).read()
open(_p,'w').write(_s.replace('v.shape.ndims is 0','v.shape.ndims == 0')
                     .replace('v.shape.ndims is 1','v.shape.ndims == 1'))

CKPT=f'{TM_STORE}/earth_texture/network-final.pkl'
assert os.path.exists(CKPT), 'checkpoint missing'
import tfutil, misc
sess=tfutil.create_session(force_as_default=True)
nets={getattr(o,'name',f'_{i}'):o for i,o in enumerate(misc.load_pkl(CKPT))}
Es_zg, Es_zl, Gs = nets['Es_zg'], nets['Es_zl'], nets['Gs']
print('LOADED OK —', Gs.output_shapes)

TM_RES, LATENT_IDX = 128, 0
def tm_prep(p):
    a=np.asarray(p.convert('RGB').resize((TM_RES,)*2),dtype=np.float32)
    return (a/127.5-1.0).transpose(2,0,1)[None]
def tm_pil(a):
    return Image.fromarray(np.clip((a[0].transpose(1,2,0)+1)*127.5,0,255).astype(np.uint8))
def tm_encode(p):
    x=tm_prep(p)
    return (Es_zg.run(x,return_as_list=True)[LATENT_IDX],
            Es_zl.run(x,return_as_list=True)[LATENT_IDX])
def tm_gen(zg,zl):
    return tm_pil(Gs.run(np.tile(zg,(1,1,zl.shape[2],zl.shape[3])),zl,return_as_list=True)[0])
def tm_interp(A,B,e=0.5):
    zgA,zlA=tm_encode(A); zgB,zlB=tm_encode(B)
    return tm_gen((1-e)*zgA+e*zgB, (1-e)*zlA+e*zlB)

In [ ]:
# ============================================================
# PHASE 1, CELL 2 — define the pair set (shared by both phases)
# and run Texture Mixer's self-loop and fixed-target loop.
# ============================================================
DTD=f'{BASE}/dtd/images'
cats=sorted([c for c in os.listdir(DTD) if os.path.isdir(f'{DTD}/{c}')])
assert len(cats)>=45, f'only {len(cats)} categories'
def safe_open(p,tries=5):
    for i in range(tries):
        try:
            im=Image.open(p); im.load(); return im.convert('RGB')
        except OSError:
            if i==tries-1: raise
            time.sleep(0.4*(i+1))

N_PAIRS, N_LOOP = 24, 8
rng=random.Random(509)
pairs=[]
for i in range(N_PAIRS):
    a,b=rng.sample(cats,2)
    pairs.append((rng.choice(sorted(glob.glob(f'{DTD}/{a}/*.jpg'))),
                  rng.choice(sorted(glob.glob(f'{DTD}/{b}/*.jpg')))))
pd.DataFrame({'k':range(N_PAIRS),'A':[p[0] for p in pairs],'B':[p[1] for p in pairs]}) \
  .to_csv(f'{DEG}/pairs.csv', index=False)
print(f'{N_PAIRS} pairs, {N_LOOP} loop iterations')

t0=time.time()
for k,(pa,pb) in enumerate(pairs):
    A,B = safe_open(pa), safe_open(pb)
    A.resize((TM_RES,)*2).save(f'{DEG}/{k:03d}_A.png')
    B.resize((TM_RES,)*2).save(f'{DEG}/{k:03d}_B.png')
    x=A                                        # self-loop: M(x,x,0.5)
    for t in range(N_LOOP):
        x=tm_interp(x,x,0.5); x.save(f'{DEG}/tm_self_{k:03d}_{t:02d}.png')
    x=A                                        # fixed-target loop: M(x,B,0.5)
    for t in range(N_LOOP):
        x=tm_interp(x,B,0.5); x.save(f'{DEG}/tm_fixed_{k:03d}_{t:02d}.png')
    if k%5==0: print(f'  {k}/{N_PAIRS} ({(time.time()-t0)/60:.1f}m)')
print('\nPHASE 1 DONE. Runtime -> Restart session, then run cells 3-5.')

In [ ]:
# ============================================================
# PHASE 2, CELL 3 — torch only. Run AFTER restarting.
# ============================================================
!pip install -q pot plenoptic einops timm
import os, sys, glob, random, time, math, json
import numpy as np, pandas as pd, torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image, ImageFile
from scipy.stats import wilcoxon
from google.colab import drive
drive.mount('/content/drive')
ImageFile.LOAD_TRUNCATED_IMAGES=True
device='cuda' if torch.cuda.is_available() else 'cpu'

BASE=os.environ.get('DATA_ROOT', '/content/drive/MyDrive/DATA_ROOT')  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
DEG=f'{BASE}/degradation_5_9'
OUT=f'{DEG}/results'; os.makedirs(OUT, exist_ok=True)
TMP='/content/tmp'; os.makedirs(TMP, exist_ok=True)
DTD_ROOT=f'{BASE}/dtd/images'
cats=sorted([c for c in os.listdir(DTD_ROOT) if os.path.isdir(f'{DTD_ROOT}/{c}')])
def safe_open(p,tries=5):
    for i in range(tries):
        try:
            im=Image.open(p); im.load(); return im.convert('RGB')
        except OSError:
            if i==tries-1: raise
            time.sleep(0.4*(i+1))
def cat_of(p): return Path(p).parent.name
print(f'{len(cats)} categories')

os.chdir('/content'); os.system('rm -rf cnt-siga24')
os.system('git clone -q https://github.com/phtu-cs/compositional-neural-textures.git cnt-siga24')
os.system('pip install -q -r cnt-siga24/requirements.in')
os.system('pip install -q "git+https://github.com/facebookresearch/detectron2.git"')
os.system(f'rm -rf /content/cnt-siga24/ckpts; ln -sfn "{BASE}/cnt_ckpts" /content/cnt-siga24/ckpts')
CNT_DIR='/content/cnt-siga24'; os.chdir(CNT_DIR); sys.path.insert(0,CNT_DIR)
from src.inference.core import TexAutoEncoderInference, load_image
model=TexAutoEncoderInference(ckpt_path=Path(f"{CNT_DIR}/ckpts/finetuned-model256.ckpt"), device=device)
IMG_SIZE=256; FMS=[round(IMG_SIZE*r) for r in model.generator.feature_map_size_ratios]

_tc=[0]
def cnt_load(x,size=IMG_SIZE):
    if isinstance(x,(str,Path)): return load_image(Path(x),size).to(device)
    _tc[0]+=1; fp=f'{TMP}/t{_tc[0]%64}.png'; x.convert('RGB').save(fp)
    return load_image(Path(fp),size).to(device)
def to_pil(d):
    a=((d.squeeze(0).detach().cpu().clamp(-1,1)+1)/2).permute(1,2,0).numpy()
    return Image.fromarray((a*255).astype(np.uint8))
def get_gen_inputs(img):
    with torch.no_grad():
        blob=model.encode(img); L=model.splat(blob)
        return blob, L["layer_level_layouts"][-1][0]["blob_features"].clone(), \
                     L["layer_level_layouts"][-1][0]["score_img"].clone()
def decode_interp(br,bf,si):
    with torch.no_grad():
        L=model.splat(br)
        L["layer_level_layouts"][-1][0]["blob_features"]=bf
        L["layer_level_layouts"][-1][0]["score_img"]=si
        return model.decode(L)
def build_style_maps(bf,si,fms=FMS):
    sp={}; s=si; sp[s.shape[-1]]=s
    while s.shape[-1]>min(fms):
        s=F.interpolate(s,s.shape[-1]//2,mode="bilinear",align_corners=False); sp[s.shape[-1]]=s
    return {sz: torch.einsum("bnc,bnhw->bchw",bf,sp[sz]) for sz in set(fms)}
def decode_with_style_maps(br,sm):
    with torch.no_grad():
        L=model.splat(br); si=L["layer_level_layouts"][-1][0]["score_img"]
        fms=[round(si.shape[-1]*r) for r in model.generator.feature_map_size_ratios]
        sp={}; s=si; sp[s.shape[-1]]=s
        while s.shape[-1]>min(fms):
            s=F.interpolate(s,s.shape[-1]//2,mode="bilinear",align_corners=False); sp[s.shape[-1]]=s
        fi=sp[fms[0]][:,1:].amax(dim=1)[:,None].repeat(1,model.generator.n_embedding,1,1)
        x=fi.clone()
        for i in range(len(fms)):
            x=model.generator.spade_blocks[i](x,sm[fms[i]])
            if i==len(fms)-1: break
            elif fms[i]!=fms[i+1]: x=F.interpolate(x,size=x.shape[-1]*2,mode="bilinear",align_corners=False)
        return torch.tanh(x.unsqueeze(1)).flatten(1,2)
def sinkhorn_transport(src,tgt,eps=0.05,n=100):
    N,M=src.shape[0],tgt.shape[0]; C=torch.cdist(src,tgt).pow(2); K=torch.exp(-C/eps)
    a=torch.ones(N,1,device=device)/N; b=torch.ones(M,1,device=device)/M; u=torch.ones(N,1,device=device)
    for _ in range(n): v=b/(K.T@u+1e-10); u=a/(K@v+1e-10)
    P=u*K*v.T; return (P/(P.sum(1,keepdim=True)+1e-10))@tgt
def build_d(ia,ib):
    ba,bfa,sia=get_gen_inputs(ia); bb,bfb,sib=get_gen_inputs(ib)
    return {"blob_a":ba,"bf_a":bfa,"si_a":sia,"blob_b":bb,"bf_b":bfb,"si_b":sib}
def interp_ot(d,eta,n_sub=4096):
    sa=build_style_maps(d["bf_a"],d["si_a"]); sb=build_style_maps(d["bf_b"],d["si_b"])
    L=max(FMS); src,tgt=sa[L][0],sb[L][0]; C,H,W=src.shape
    sf=src.permute(1,2,0).reshape(-1,C); tf=tgt.permute(1,2,0).reshape(-1,C); N=sf.shape[0]
    i_s=torch.randperm(N,device=device)[:n_sub]; i_t=torch.randperm(N,device=device)[:n_sub]
    with torch.no_grad(): tt=sinkhorn_transport(sf[i_s],tf[i_t])
    nn=torch.cdist(sf,sf[i_s]).argmin(1)
    it=((1-eta)*sf+eta*tt[nn]).reshape(H,W,C).permute(2,0,1).unsqueeze(0)
    return decode_with_style_maps(d["blob_a"],{s:(it if s==L else F.interpolate(it,size=s,mode="bilinear",align_corners=False)) for s in set(FMS)})
def interp_ot_sym(d,eta,n_sub=4096):
    sa=build_style_maps(d["bf_a"],d["si_a"]); sb=build_style_maps(d["bf_b"],d["si_b"])
    L=max(FMS); A,B=sa[L][0],sb[L][0]; C,H,W=A.shape
    af=A.permute(1,2,0).reshape(-1,C); bf=B.permute(1,2,0).reshape(-1,C); N=af.shape[0]
    ia=torch.randperm(N,device=device)[:n_sub]; ib=torch.randperm(N,device=device)[:n_sub]
    with torch.no_grad():
        tab=sinkhorn_transport(af[ia],bf[ib]); tba=sinkhorn_transport(bf[ib],af[ia])
    na=torch.cdist(af,af[ia]).argmin(1); nb=torch.cdist(bf,bf[ib]).argmin(1)
    it=(0.5*(((1-eta)*af+eta*tab[na])+(eta*bf+(1-eta)*tba[nb]))).reshape(H,W,C).permute(2,0,1).unsqueeze(0)
    return decode_with_style_maps(d["blob_a"],{s:(it if s==L else F.interpolate(it,size=s,mode="bilinear",align_corners=False)) for s in set(FMS)})
def interp_lin_pixel(d,eta):
    sa=build_style_maps(d["bf_a"],d["si_a"]); sb=build_style_maps(d["bf_b"],d["si_b"])
    return decode_with_style_maps(d["blob_a"],{s:(1-eta)*sa[s]+eta*sb[s] for s in set(FMS)})
def interp_lin_texton(d,eta):
    return decode_interp(d["blob_a"],(1-eta)*d["bf_a"]+eta*d["bf_b"],(1-eta)*d["si_a"]+eta*d["si_b"])
FNS={'ot':interp_ot,'ot_sym':interp_ot_sym,
     'pixel_linear':interp_lin_pixel,'texton_linear':interp_lin_texton}
def our_interp(m,A,B,eta=0.5):
    return to_pil(FNS[m](build_d(cnt_load(A),cnt_load(B)),eta))

# --- observers ---
import timm
dino=timm.create_model('vit_small_patch14_dinov2.lvd142m',pretrained=True,num_classes=0).to(device).eval()
for p in dino.parameters(): p.requires_grad_(False)
_M=torch.tensor([0.485,0.456,0.406],device=device).view(1,3,1,1)
_S=torch.tensor([0.229,0.224,0.225],device=device).view(1,3,1,1)
@torch.no_grad()
def dino_feats(pils,bs=16):
    out=[]
    for i in range(0,len(pils),bs):
        xs=[torch.from_numpy(np.asarray(p.convert('RGB').resize((256,256))).copy()).float().permute(2,0,1)/255. for p in pils[i:i+bs]]
        b=torch.stack(xs).to(device); b=F.interpolate(b,size=518,mode='bilinear',align_corners=False); b=(b-_M)/_S
        out.append(dino(b).cpu().numpy())
    return np.concatenate(out)
from torchvision.models import inception_v3, Inception_V3_Weights
_inc=inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1,aux_logits=True).to(device).eval()
_inc.fc=torch.nn.Identity()
for p in _inc.parameters(): p.requires_grad_(False)
@torch.no_grad()
def inc_feats(pils,bs=24):
    out=[]
    for i in range(0,len(pils),bs):
        xs=[torch.from_numpy(np.asarray(p.convert('RGB').resize((256,256))).copy()).float().permute(2,0,1)/255. for p in pils[i:i+bs]]
        b=torch.stack(xs).to(device); b=F.interpolate(b,size=299,mode='bilinear',align_corners=False); b=(b-_M)/_S
        out.append(_inc(b).cpu().numpy())
    return np.concatenate(out)
import plenoptic as po
ps_model=po.models.PortillaSimoncelli((IMG_SIZE,IMG_SIZE),n_scales=4,n_orientations=4,spatial_corr_width=7).to(device)
def _raw_ps(p):
    g=p.convert('L').resize((IMG_SIZE,)*2)
    x=torch.from_numpy(np.asarray(g).copy()).float()[None,None].to(device)/255.
    with torch.no_grad(): return ps_model(x).detach().cpu().numpy().ravel()
VALID=~np.isnan(_raw_ps(Image.effect_noise((256,256),60)))
def ps_stats(p): return _raw_ps(p)[VALID]

def manifold(featfn,K=3,n=400,seed=0):
    rr=random.Random(seed); imgs=[]
    for i in range(n):
        try: imgs.append(safe_open(rr.choice(sorted(glob.glob(f'{DTD_ROOT}/{rr.choice(cats)}/*.jpg')))).resize((IMG_SIZE,)*2))
        except OSError: continue
    Phi=featfn(imgs); Phi=Phi[~np.isnan(Phi).any(axis=1)]
    D=np.linalg.norm(Phi[:,None]-Phi[None,:],axis=2); np.fill_diagonal(D,np.inf)
    rad=np.sort(D,axis=1)[:,K-1]; keep=rad<=np.median(rad)
    Pr,rd=Phi[keep],rad[keep]
    return lambda f: float(np.max(rd/(np.linalg.norm(Pr-f,axis=1)+1e-9)))
print('\nPHASE 2 READY')

In [ ]:
# ============================================================
# CELL 4 — degradation under iteration.
#   self loop : x <- M(x, x, 0.5)   [M(A,A)=A is an axiom -> all drift is decay]
#   fixed loop: x <- M(x, B, 0.5)   [B held fixed; should converge to B]
# Realism is averaged over 3 manifold draws, since a single manifold
# has ~0.018 std (measured earlier).
# ============================================================
pairs=pd.read_csv(f'{DEG}/pairs.csv'); N_LOOP=8
R_DINO=[manifold(dino_feats,seed=s) for s in range(3)]
R_INC =[manifold(inc_feats, seed=s) for s in range(3)]
def realism_d(f): return float(np.mean([r(f) for r in R_DINO]))
def realism_i(f): return float(np.mean([r(f) for r in R_INC]))
print('manifolds built (3 draws each)')

METHODS=list(FNS)+['texture_mixer']
rows=[]; t0=time.time()
for _,row in pairs.iterrows():
    k=int(row.k)
    A=Image.open(f'{DEG}/{k:03d}_A.png').convert('RGB').resize((IMG_SIZE,)*2)
    B=Image.open(f'{DEG}/{k:03d}_B.png').convert('RGB').resize((IMG_SIZE,)*2)
    fA,fB=dino_feats([A])[0],dino_feats([B])[0]
    psA=ps_stats(A)

    for loop in ['self','fixed']:
        for m in METHODS:
            if m=='texture_mixer':
                fs=sorted(glob.glob(f'{DEG}/tm_{loop}_{k:03d}_*.png'))
                if not fs: continue
                frames=[Image.open(f).convert('RGB').resize((IMG_SIZE,)*2) for f in fs]
            else:
                frames=[]; x=A
                for t in range(N_LOOP):
                    x=our_interp(m, x, x if loop=='self' else B, 0.5)
                    frames.append(x)
                if k<2:
                    for t,f in enumerate(frames): f.save(f'{OUT}/{loop}_{m}_{k:03d}_{t:02d}.png')
            Fm  = dino_feats(frames)
            Fmi = inc_feats(frames)
            rd = [realism_d(f) for f in Fm]
            ri = [realism_i(f) for f in Fmi]

            rec={'pair':k,'loop':loop,'method':m,'n':len(frames),
                 'realism_0':rd[0],'realism_N':rd[-1],'drop_dino':rd[0]-rd[-1],
                 'drop_inc':ri[0]-ri[-1],
                 'drift_per_step':float(np.mean(np.linalg.norm(np.diff(Fm,axis=0),axis=1))),
                 'ps_drift':float(np.linalg.norm(ps_stats(frames[-1])-psA)),
                 'traj':json.dumps(rd)}
            if loop=='self':
                rec['drift_from_A']=float(np.linalg.norm(Fm[-1]-fA))
            else:
                rec['dist_B_first']=float(np.linalg.norm(Fm[0]-fB))
                rec['dist_B_last'] =float(np.linalg.norm(Fm[-1]-fB))
            rows.append(rec)
    print(f'  pair {k}/{len(pairs)} ({(time.time()-t0)/60:.1f}m)')

dfd=pd.DataFrame(rows); dfd.to_csv(f'{OUT}/degradation.csv',index=False)

for loop,desc in [('self','M(x,x,0.5) — identity axiom, all drift is decay'),
                  ('fixed','M(x,B,0.5) — should converge to B and stay real')]:
    s=dfd[dfd.loop==loop]
    if not len(s): continue
    print(f"\n=== {loop.upper()} LOOP: {desc} (n={s.pair.nunique()}) ===")
    hdr=f"  {'method':16} {'drop DINO':>11} {'drop Inc':>10} {'drift/step':>11} {'PS drift':>10}"
    hdr += f" {'drift from A':>13}" if loop=='self' else f" {'converged':>10}"
    print(hdr)
    for m in METHODS:
        t=s[s.method==m]
        if not len(t): continue
        line=(f"  {m:16} {t.drop_dino.median():11.4f} {t.drop_inc.median():10.4f} "
              f"{t.drift_per_step.median():11.2f} {t.ps_drift.median():10.3f}")
        line += (f" {t.drift_from_A.median():13.2f}" if loop=='self'
                 else f" {100*(t.dist_B_last<t.dist_B_first).mean():9.0f}%")
        print(line)

print(f"\n=== PAIRED: OT vs each baseline (DINO realism drop; lower better) ===")
for loop in ['self','fixed']:
    print(f"  [{loop} loop]")
    for base in [b for b in METHODS if b!='ot']:
        a=dfd[(dfd.method=='ot')&(dfd.loop==loop)].set_index('pair').drop_dino
        b=dfd[(dfd.method==base)&(dfd.loop==loop)].set_index('pair').drop_dino
        ix=a.index.intersection(b.index)
        if len(ix)<5: continue
        d=a[ix]-b[ix]
        p=wilcoxon(d).pvalue if d.abs().sum()>0 else 1.0
        print(f"    ot vs {base:15} wins {int((d<0).sum()):2d}/{len(d)}  "
              f"median diff {d.median():+.4f}  p={p:.3g}")

In [ ]:
# ============================================================
# CELL 5 — trajectory plot + filmstrip
# ============================================================
fig,ax=plt.subplots(1,2,figsize=(13.5,4.4))
for j,loop in enumerate(['self','fixed']):
    for m in METHODS:
        s=dfd[(dfd.method==m)&(dfd.loop==loop)]
        tr=np.array([json.loads(t) for t in s.traj if len(json.loads(t))>=N_LOOP])
        if not len(tr): continue
        med=np.median(tr,axis=0); q1=np.percentile(tr,25,axis=0); q3=np.percentile(tr,75,axis=0)
        ax[j].plot(range(1,len(med)+1),med,marker='o',label=m,lw=2)
        ax[j].fill_between(range(1,len(med)+1),q1,q3,alpha=0.12)
    ax[j].set_xlabel('loop iteration'); ax[j].set_ylabel('DINOv2 realism')
    ax[j].set_title(f'{loop} loop: '+('M(x,x,0.5)' if loop=='self' else 'M(x,B,0.5)'))
    ax[j].grid(alpha=.25); ax[j].legend(fontsize=8)
plt.suptitle('Does the operator survive repeated application?',fontsize=13)
plt.tight_layout(); plt.savefig(f'{OUT}/trajectories.png',dpi=180,bbox_inches='tight'); plt.show()

SHOW=['ot','pixel_linear','texture_mixer']
for loop in ['self','fixed']:
    fig,ax=plt.subplots(len(SHOW),N_LOOP+1,figsize=(1.65*(N_LOOP+1),1.85*len(SHOW)))
    for i,m in enumerate(SHOW):
        pat=(f'{OUT}/{loop}_{m}_000_*.png' if m!='texture_mixer'
             else f'{DEG}/tm_{loop}_000_*.png')
        fs=sorted(glob.glob(pat))
        ax[i,0].imshow(Image.open(f'{DEG}/000_A.png')); ax[i,0].axis('off')
        ax[i,0].text(-0.1,0.5,m,transform=ax[i,0].transAxes,ha='right',va='center',fontsize=9)
        if i==0: ax[i,0].set_title('input',fontsize=8)
        for j in range(N_LOOP):
            ax[i,j+1].axis('off')
            if j<len(fs):
                ax[i,j+1].imshow(Image.open(fs[j]))
                if i==0: ax[i,j+1].set_title(f'{j+1}',fontsize=8)
    plt.suptitle(f'{loop} loop, pair 0',fontsize=12)
    plt.tight_layout(); plt.savefig(f'{OUT}/strip_{loop}.png',dpi=180,bbox_inches='tight'); plt.show()

In [ ]:
# How much drift comes from the autoencoder alone?
rows_ae=[]
for _,row in pairs.iloc[:12].iterrows():
    k=int(row.k)
    A=Image.open(f'{DEG}/{k:03d}_A.png').convert('RGB').resize((IMG_SIZE,)*2)
    x=A; frames=[]
    for t in range(8):
        x=to_pil(model.decode(model.splat(model.encode(cnt_load(x))))); frames.append(x)
    Fm=dino_feats(frames); fA=dino_feats([A])[0]
    rows_ae.append({'pair':k,'drift_from_A':float(np.linalg.norm(Fm[-1]-fA)),
                    'realism_drop':realism_d(Fm[0])-realism_d(Fm[-1])})
ae=pd.DataFrame(rows_ae)
print(f"autoencoder-only loop: drift {ae.drift_from_A.median():.1f}, "
      f"realism drop {ae.realism_drop.median():+.4f}")
print(f"  vs OT self-loop drift {dfd[(dfd.method=='ot')&(dfd.loop=='self')].drift_from_A.median():.1f}")

In [ ]:
# ============================================================
# CONTROL FILMSTRIP: pure encode->decode, 8 cycles, no interpolation.
# This is the drift every method inherits before its operator acts.
# ============================================================
SHOW_PAIRS = [0, 1, 2]
N_LOOP = 8

fig, ax = plt.subplots(len(SHOW_PAIRS), N_LOOP+1, figsize=(1.65*(N_LOOP+1), 1.85*len(SHOW_PAIRS)))
ax = np.atleast_2d(ax)
ae_rows = []
for r, k in enumerate(SHOW_PAIRS):
    A = Image.open(f'{DEG}/{k:03d}_A.png').convert('RGB').resize((IMG_SIZE,)*2)
    ax[r,0].imshow(A); ax[r,0].axis('off')
    if r == 0: ax[r,0].set_title('input', fontsize=9)
    x = A; frames = []
    for t in range(N_LOOP):
        x = to_pil(model.decode(model.splat(model.encode(cnt_load(x)))))
        frames.append(x)
        x.save(f'{OUT}/autoenc_{k:03d}_{t:02d}.png')
        ax[r,t+1].imshow(x); ax[r,t+1].axis('off')
        if r == 0: ax[r,t+1].set_title(f'{t+1}', fontsize=9)
    Fm = dino_feats(frames); fA = dino_feats([A])[0]
    ae_rows.append({'pair': k,
                    'drift': float(np.linalg.norm(Fm[-1]-fA)),
                    'realism_0': realism_d(Fm[0]), 'realism_N': realism_d(Fm[-1]),
                    'ps_drift': float(np.linalg.norm(ps_stats(frames[-1]) - ps_stats(A)))})
plt.suptitle('Control: encode → decode, 8 cycles, no interpolation', fontsize=12)
plt.tight_layout(); plt.savefig(f'{OUT}/strip_autoencoder.png', dpi=180, bbox_inches='tight'); plt.show()

for rec in ae_rows:
    print(f"  pair {rec['pair']}: drift {rec['drift']:.1f}  "
          f"realism {rec['realism_0']:.3f} -> {rec['realism_N']:.3f}  "
          f"PS drift {rec['ps_drift']:.2f}")

In [ ]:
# ============================================================
# THREE-WAY CONTROL STRIP — same input, three loops:
#   (a) CNT encode->decode only      (no operator)
#   (b) OT self-loop: M(x,x,0.5)     (transport x onto itself)
#   (c) Texture Mixer self-loop      (their encoder/decoder cycle)
# Isolates how much drift belongs to the representation vs the operator.
# ============================================================
SHOW_PAIRS = [0, 1]
N_LOOP = 8
ROWS = ['CNT autoencoder\n(no operator)', 'OT  M(x,x,0.5)', 'Texture Mixer\nM(x,x,0.5)']

summary = []
for k in SHOW_PAIRS:
    A = Image.open(f'{DEG}/{k:03d}_A.png').convert('RGB').resize((IMG_SIZE,)*2)
    fA = dino_feats([A])[0]; psA = ps_stats(A)

    # (a) autoencoder only
    x = A; ae = []
    for t in range(N_LOOP):
        x = to_pil(model.decode(model.splat(model.encode(cnt_load(x)))))
        ae.append(x)
    # (b) OT self-loop
    x = A; ot = []
    for t in range(N_LOOP):
        x = our_interp('ot', x, x, 0.5); ot.append(x)
    # (c) Texture Mixer self-loop (from Phase 1)
    tm = [Image.open(f).convert('RGB').resize((IMG_SIZE,)*2)
          for f in sorted(glob.glob(f'{DEG}/tm_self_{k:03d}_*.png'))]

    fig, ax = plt.subplots(3, N_LOOP+1, figsize=(1.6*(N_LOOP+1), 1.85*3))
    for r, (lbl, seq) in enumerate(zip(ROWS, [ae, ot, tm])):
        ax[r,0].imshow(A); ax[r,0].axis('off')
        ax[r,0].text(-0.08, 0.5, lbl, transform=ax[r,0].transAxes,
                     ha='right', va='center', fontsize=8)
        if r == 0: ax[r,0].set_title('input', fontsize=9)
        for t in range(N_LOOP):
            ax[r,t+1].axis('off')
            if t < len(seq):
                ax[r,t+1].imshow(seq[t])
                if r == 0: ax[r,t+1].set_title(f'{t+1}', fontsize=9)
        if seq:
            Fm = dino_feats(seq)
            summary.append({'pair':k, 'loop':lbl.split('\n')[0],
                            'drift': float(np.linalg.norm(Fm[-1]-fA)),
                            'realism_0': realism_d(Fm[0]), 'realism_N': realism_d(Fm[-1]),
                            'realism_drop': realism_d(Fm[0])-realism_d(Fm[-1]),
                            'ps_drift': float(np.linalg.norm(ps_stats(seq[-1])-psA))})
    plt.suptitle(f'pair {k}: what does the representation do on its own?', fontsize=12)
    plt.tight_layout(); plt.savefig(f'{OUT}/strip_threeway_{k:03d}.png', dpi=180, bbox_inches='tight')
    plt.show()

s = pd.DataFrame(summary)
print("\n=== DRIFT ATTRIBUTION (median over shown pairs) ===")
print(f"  {'loop':22} {'DINO drift':>11} {'realism drop':>13} {'PS drift':>10}")
for L in s.loop.unique():
    t = s[s.loop==L]
    print(f"  {L:22} {t.drift.median():11.1f} {t.realism_drop.median():13.4f} {t.ps_drift.median():10.2f}")
print("\n  If OT ≈ autoencoder, the operator contributes almost nothing to the drift.")

In [ ]:
# ============================================================
# Is the contraction OT's or CNT's? Iterate transport in DINO
# feature space with no decoder in the loop.
# ============================================================
@torch.no_grad()
def dino_patch_cloud(pil):
    """Per-patch DINOv2 features: a cloud, like CNT's style map."""
    x = torch.from_numpy(np.asarray(pil.convert('RGB').resize((256,256))).copy()) \
             .float().permute(2,0,1)[None].to(device)/255.
    x = F.interpolate(x, size=518, mode='bilinear', align_corners=False); x = (x-_M)/_S
    f = dino.forward_features(x)            # [1, 1+N, C]
    return f[0, 1:, :]                      # drop CLS -> [N, C]

for k in [0, 1]:
    A = Image.open(f'{DEG}/{k:03d}_A.png').convert('RGB').resize((IMG_SIZE,)*2)
    cloud = dino_patch_cloud(A)
    v0 = float(cloud.var(0).mean())
    print(f'\npair {k}: DINO patch cloud {tuple(cloud.shape)}, initial var {v0:.4f}')
    x = cloud.clone()
    for t in range(8):
        n = x.shape[0]; sub = min(2048, n)
        i_s = torch.randperm(n, device=device)[:sub]
        i_t = torch.randperm(n, device=device)[:sub]
        tt = sinkhorn_transport(x[i_s], x[i_t])          # transport onto itself
        nn = torch.cdist(x, x[i_s]).argmin(1)
        x = 0.5*x + 0.5*tt[nn]                            # eta = 0.5
        print(f'  iter {t+1}: var {float(x.var(0).mean()):.4f} '
              f'({100*float(x.var(0).mean())/v0:.0f}% of start)')

# same thing on the CNT style-map cloud, for comparison
for k in [0, 1]:
    A = Image.open(f'{DEG}/{k:03d}_A.png').convert('RGB').resize((IMG_SIZE,)*2)
    _, bf, si = get_gen_inputs(cnt_load(A))
    sm = build_style_maps(bf, si)[max(FMS)][0]
    x = sm.permute(1,2,0).reshape(-1, sm.shape[0])
    v0 = float(x.var(0).mean())
    print(f'\npair {k}: CNT style cloud {tuple(x.shape)}, initial var {v0:.4f}')
    for t in range(8):
        n = x.shape[0]
        i_s = torch.randperm(n, device=device)[:4096]
        i_t = torch.randperm(n, device=device)[:4096]
        tt = sinkhorn_transport(x[i_s], x[i_t])
        nn = torch.cdist(x, x[i_s]).argmin(1)
        x = 0.5*x + 0.5*tt[nn]
        print(f'  iter {t+1}: var {float(x.var(0).mean()):.4f} '
              f'({100*float(x.var(0).mean())/v0:.0f}% of start)')

In [ ]:
# Is eps=0.05 uniform-smoothing on CNT and near-identity on DINO?
for name, cloud in [('DINO', dino_patch_cloud(Image.open(f'{DEG}/000_A.png').convert('RGB'))),
                    ('CNT',  (lambda: (lambda sm: sm.permute(1,2,0).reshape(-1, sm.shape[0]))(
                        build_style_maps(*get_gen_inputs(cnt_load(
                            Image.open(f'{DEG}/000_A.png').convert('RGB')))[1:])[max(FMS)][0]))())]:
    sub = cloud[torch.randperm(cloud.shape[0], device=device)[:2048]]
    C = torch.cdist(sub, sub).pow(2)
    print(f'{name}: cost median {float(C.median()):.3e}, '
          f'ratio to eps=0.05 -> {float(C.median())/0.05:.2e}')
    K = torch.exp(-C/0.05)
    P = K/K.sum(1, keepdim=True)
    print(f'   plan row entropy {float(-(P*torch.log(P+1e-12)).sum(1).mean()):.2f} '
          f'(0 = permutation, {math.log(2048):.2f} = uniform)')

In [ ]:
# Same pairs, same everything, only eps scaled relative to the cost.
def sinkhorn_transport_rel(src, tgt, eps_rel=0.05, n=100):
    C = torch.cdist(src, tgt).pow(2)
    eps = eps_rel * C.median().clamp_min(1e-12)          # <-- relative
    K = torch.exp(-C/eps)
    N, M = src.shape[0], tgt.shape[0]
    a = torch.ones(N,1,device=device)/N; b = torch.ones(M,1,device=device)/M
    u = torch.ones(N,1,device=device)
    for _ in range(n): v = b/(K.T@u+1e-10); u = a/(K@v+1e-10)
    P = u*K*v.T
    ent = float(-(P/P.sum(1,keepdim=True)*torch.log(P/P.sum(1,keepdim=True)+1e-12)).sum(1).mean())
    return (P/(P.sum(1,keepdim=True)+1e-10))@tgt, ent

A = Image.open(f'{DEG}/000_A.png').convert('RGB').resize((IMG_SIZE,)*2)
B = Image.open(f'{DEG}/000_B.png').convert('RGB').resize((IMG_SIZE,)*2)
d = build_d(cnt_load(A), cnt_load(B))
sa = build_style_maps(d["bf_a"], d["si_a"]); sb = build_style_maps(d["bf_b"], d["si_b"])
L = max(FMS); C_,H,W = sa[L][0].shape
sf = sa[L][0].permute(1,2,0).reshape(-1,C_); tf = sb[L][0].permute(1,2,0).reshape(-1,C_)
i_s = torch.randperm(sf.shape[0], device=device)[:4096]
i_t = torch.randperm(tf.shape[0], device=device)[:4096]

fig, ax = plt.subplots(1, 6, figsize=(17, 3.2))
ax[0].imshow(A); ax[0].axis('off'); ax[0].set_title('A', fontsize=10)
for c, er in enumerate([0.5, 0.1, 0.02, 0.005]):
    tt, ent = sinkhorn_transport_rel(sf[i_s], tf[i_t], eps_rel=er)
    nn = torch.cdist(sf, sf[i_s]).argmin(1)
    it = (0.5*sf + 0.5*tt[nn]).reshape(H,W,C_).permute(2,0,1).unsqueeze(0)
    im = to_pil(decode_with_style_maps(d["blob_a"],
          {s:(it if s==L else F.interpolate(it,size=s,mode="bilinear",align_corners=False)) for s in set(FMS)}))
    ax[c+1].imshow(im); ax[c+1].axis('off')
    ax[c+1].set_title(f'eps_rel={er}\nentropy {ent:.2f}', fontsize=9)
ax[5].imshow(B); ax[5].axis('off'); ax[5].set_title('B', fontsize=10)
plt.suptitle('midpoint vs entropic regularisation (uniform ceiling = 8.32)', fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# Does the transported cloud's variance actually change with eps?
for er in [0.5, 0.1, 0.02, 0.005]:
    tt, ent = sinkhorn_transport_rel(sf[i_s], tf[i_t], eps_rel=er)
    v_tt = float(tt.var(0).mean()); v_tf = float(tf[i_t].var(0).mean())
    print(f'eps_rel={er:<6} entropy {ent:.2f} | transported var {v_tt:.5f} '
          f'| target var {v_tf:.5f} | retained {100*v_tt/v_tf:.0f}%')

# And what does the nn-expansion cost?
nn = torch.cdist(sf, sf[i_s]).argmin(1)
tt, _ = sinkhorn_transport_rel(sf[i_s], tf[i_t], eps_rel=0.005)
v_before = float(tt.var(0).mean())
v_after  = float(tt[nn].var(0).mean())
print(f'\nafter nn expansion: var {v_after:.5f} (vs {v_before:.5f} before)')
print(f'unique targets used: {len(torch.unique(nn))} of {len(sf)} grid points')

In [ ]:
x = Image.open(f'{DEG}/001_A.png').convert('RGB').resize((IMG_SIZE,)*2)
frames = []
for t in range(8):
    d = build_d(cnt_load(x), cnt_load(x))
    sa = build_style_maps(d["bf_a"], d["si_a"]); L = max(FMS)
    s = sa[L][0]; C_,H,W = s.shape
    sf_ = s.permute(1,2,0).reshape(-1,C_)
    i1 = torch.randperm(sf_.shape[0], device=device)[:4096]
    i2 = torch.randperm(sf_.shape[0], device=device)[:4096]
    tt,_ = sinkhorn_transport_rel(sf_[i1], sf_[i2], eps_rel=0.005)
    nn = torch.cdist(sf_, sf_[i1]).argmin(1)
    it = (0.5*sf_ + 0.5*tt[nn]).reshape(H,W,C_).permute(2,0,1).unsqueeze(0)
    x = to_pil(decode_with_style_maps(d["blob_a"],
        {s_:(it if s_==L else F.interpolate(it,size=s_,mode="bilinear",align_corners=False)) for s_ in set(FMS)}))
    frames.append(x)
fig, ax = plt.subplots(1, 9, figsize=(15,2))
ax[0].imshow(Image.open(f'{DEG}/001_A.png')); ax[0].axis('off'); ax[0].set_title('input', fontsize=8)
for i,f in enumerate(frames):
    ax[i+1].imshow(f); ax[i+1].axis('off'); ax[i+1].set_title(f'{i+1}', fontsize=8)
plt.suptitle('self-loop at eps_rel=0.005 — does sharper transport stop the collapse?', fontsize=11)
plt.tight_layout(); plt.show()